# Activation Functions

> The non-linear transformations that give neural networks the ability to learn complex patterns — without them, a deep network is mathematically equivalent to a single linear layer.

---

## Introduction

In an artificial neural network, each neuron computes a **weighted sum** of its inputs plus a bias:

$$z = w_1x_1 + w_2x_2 + \cdots + w_nx_n + b = \mathbf{w}^T\mathbf{x} + b$$

This scalar $z$ is then passed through an **activation function** $f$:

$$a = f(z)$$

The output $a$ is what gets passed to the next layer. The activation function decides whether and how strongly a neuron "fires" — and crucially, it introduces non-linearity into the network.

---

## Why Are Activation Functions Needed?

Without an activation function, every layer would just compute a linear transformation. No matter how many layers you stack, the composition of linear functions is still a linear function:

$$W_3(W_2(W_1 x)) = (W_3 W_2 W_1)\, x = W_{combined}\, x$$

A network without activation functions — regardless of depth — can only model linear relationships. It would be no more powerful than logistic regression.

Activation functions introduce **non-linearity**, allowing the network to approximate arbitrarily complex functions and learn hierarchical representations from data. This is the theoretical basis of the **Universal Approximation Theorem**: a neural network with at least one hidden layer and a non-linear activation function can approximate any continuous function to arbitrary precision.

---

## Properties of an Ideal Activation Function

### 1. Non-linear
Enables the network to learn complex, non-linear mappings from inputs to outputs. Without this, depth provides no additional expressive power.

### 2. Differentiable
Backpropagation requires computing gradients of the loss with respect to each weight. If the activation function is not differentiable, gradients cannot flow backward through the network.

### 3. Computationally Inexpensive
Activation functions are evaluated millions of times per training step across all neurons in all layers. Functions involving expensive operations (e.g., exponentials in large quantities) slow training significantly.

### 4. Zero-centered
When activations are centered around zero, the gradients of the weights in the next layer have mixed signs (positive and negative), allowing more efficient and stable gradient descent updates. Non-zero-centered outputs cause all gradients to be the same sign, leading to inefficient zigzag weight updates.

### 5. Non-saturating
A saturating function produces near-constant output for large positive or negative inputs — causing gradients in those regions to approach zero. This is the source of the **vanishing gradient problem**: gradients become so small that early layers in a deep network receive almost no learning signal.

---

## Sigmoid Function

$$\sigma(z) = \frac{1}{1 + e^{-z}}$$

The sigmoid squashes any real-valued input into the range $(0, 1)$, which made it historically popular for binary classification output layers and early hidden layers.

Its derivative is:

$$\sigma'(z) = \sigma(z)(1 - \sigma(z))$$

The maximum value of this derivative is $0.25$, occurring at $z = 0$.

### Advantages
- Output bounded in $(0, 1)$ — interpretable as a probability
- Smooth and differentiable everywhere
- Non-linear

### Disadvantages

**Vanishing Gradients:** For inputs with large magnitude (positive or negative), the sigmoid saturates — output approaches 0 or 1, and the gradient approaches 0. During backpropagation, this near-zero gradient is multiplied through every layer. In a deep network, gradients shrink exponentially with depth, and early layers receive almost no update signal — training effectively stalls.

**Non-zero centered:** Sigmoid outputs are always positive (between 0 and 1). This means the gradients passed to the previous layer are always the same sign as the incoming error signal. During gradient descent, all weight updates in a layer are either all positive or all negative — the optimizer cannot move in arbitrary directions and must zigzag toward the optimum, slowing convergence.

**Computationally expensive:** Involves an exponential operation evaluated at every neuron.

### When to Use
- Output layer for **binary classification** (produces probabilities)
- Avoid in hidden layers of deep networks due to vanishing gradients

---

## Tanh (Hyperbolic Tangent)

$$\tanh(z) = \frac{e^z - e^{-z}}{e^z + e^{-z}}$$

Tanh is a rescaled version of the sigmoid, squashing inputs to the range $(-1, 1)$. Its derivative is:

$$\tanh'(z) = 1 - \tanh^2(z)$$

The maximum gradient is $1.0$ at $z = 0$, which is four times larger than sigmoid's maximum gradient of $0.25$.


### Advantages
- **Zero-centered** — outputs range from $-1$ to $1$, enabling mixed-sign gradients and more efficient weight updates than sigmoid
- Stronger gradients than sigmoid — less severe vanishing gradient problem
- Smooth and differentiable everywhere
- Non-linear

### Disadvantages
- Still a **saturating function** — vanishing gradient problem persists for large $|z|$
- More computationally expensive than ReLU (involves two exponentials)
- Not suitable for very deep networks where gradient flow across many layers is critical

### When to Use
- Hidden layers in **shallow networks** where the zero-centered property matters
- Recurrent networks (LSTMs use tanh internally)
- Preferred over sigmoid for hidden layers when a bounded output is needed

---

## ReLU (Rectified Linear Unit)

$$\text{ReLU}(z) = \max(0, z)$$

ReLU outputs the input directly if it is positive, and zero otherwise. Despite its simplicity, it is the most widely used activation function in modern deep learning.

Its derivative is:

$$\text{ReLU}'(z) = \begin{cases} 1 & \text{if } z > 0 \\ 0 & \text{if } z \leq 0 \end{cases}$$


### Advantages
- **Non-saturating for positive inputs** — gradient is always 1 for $z > 0$, so gradients flow cleanly through deep networks without shrinking
- **Computationally cheap** — just a threshold comparison, no exponentials
- **Sparse activation** — negative inputs produce zero output, meaning roughly half the neurons are inactive at any time, reducing computation and acting as a mild regularizer
- Empirically outperforms sigmoid and tanh in deep networks

### Disadvantages

**Dying ReLU Problem:** If a neuron's input is consistently negative — due to a large negative bias or an unfortunate weight initialization — its output and gradient are always zero. That neuron never updates and becomes permanently inactive ("dead"). With a high learning rate, a large fraction of neurons can die, effectively reducing model capacity.

**Non-zero centered:** ReLU outputs are always $\geq 0$, sharing the non-zero centered drawback of sigmoid.

**Unbounded output:** Unlike sigmoid and tanh, ReLU has no upper bound, which can cause exploding activations without proper weight initialization or batch normalization.

### When to Use
- Default choice for **hidden layers** in most feedforward and convolutional networks
- Deep networks where gradient flow across many layers is essential

---

## Variants of ReLU

| Variant | Formula | Addresses |
|---|---|---|
| **Leaky ReLU** | $\max(\alpha z, z)$, $\alpha = 0.01$ | Dying ReLU — small negative slope keeps neurons alive |
| **PReLU** | $\max(\alpha z, z)$, $\alpha$ learned | Dying ReLU — slope is a learned parameter |
| **ELU** | $z$ if $z>0$; $\alpha(e^z - 1)$ if $z \leq 0$ | Zero-centered output, smooth for negatives |
| **GELU** | $z \cdot \Phi(z)$ (Gaussian CDF) | Used in Transformers (BERT, GPT); smooth approximation |
| **Swish** | $z \cdot \sigma(z)$ | Smooth, non-monotonic; used in EfficientNet |

---

## Comparison Summary

| Function | Range | Zero-centered | Saturating | Gradient (max) | Use Case |
|---|---|---|---|---|---|
| Sigmoid | $(0, 1)$ | No | Yes | 0.25 | Binary output layer |
| Tanh | $(-1, 1)$ | Yes | Yes | 1.0 | Shallow hidden layers, RNNs |
| ReLU | $[0, \infty)$ | No | No (positive) | 1.0 | Deep hidden layers (default) |
| Leaky ReLU | $(-\infty, \infty)$ | No | No | 1.0 | When dying ReLU is a concern |
| GELU | $\approx(-0.17, \infty)$ | No | No | ~1.0 | Transformers |

---

## Key Takeaway

> Use **ReLU** (or a variant like Leaky ReLU or GELU) as the default for hidden layers in deep networks — it is fast, non-saturating, and empirically effective. Reserve **Sigmoid** for binary classification output layers and **Tanh** for recurrent architectures. Avoid Sigmoid and Tanh in deep hidden layers due to the vanishing gradient problem.